In [105]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import torch
from transformers import AutoTokenizer, AutoModel 
from deutschland import klinikatlas
from deutschland.klinikatlas.api import default_api
from deutschland.klinikatlas.model.fileadmin_json_icd_codes_json_get200_response_inner import (
    FileadminJsonIcdCodesJsonGet200ResponseInner as klinikatlas_datatype
)
from transformers import XLMRobertaTokenizer, XLMRobertaModel


In [106]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)


cuda


In [107]:
BASE_URL = "https://bundes-klinik-atlas.de"

configuration = klinikatlas.Configuration(
    host=BASE_URL
)

with klinikatlas.ApiClient(configuration) as api_client:
    api = default_api.DefaultApi(api_client)

    icd_data = api.fileadmin_json_icd_codes_json_get()

In [108]:
tokenizer = AutoTokenizer.from_pretrained("permediq/SapBERT-DE", use_fast=True)
model = AutoModel.from_pretrained("permediq/SapBERT-DE").to(device)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [109]:
def preprocess_input_string(input_str:str| list[str])->list[dict]:
    input_str = input_str.split() if isinstance(input_str, str) else input_str
        
    list_preprocessed_string = []
    for value in input_str:
        list_preprocessed_string.append({'description':value})
    return list_preprocessed_string

In [110]:
def german_medical_embedding (data: list[klinikatlas_datatype]|list[dict]|str, 
                              bs:int, 
                              tokenizer:XLMRobertaTokenizer, 
                              model:XLMRobertaModel) -> torch.Tensor:
    data = preprocess_input_string(data) if isinstance(data, str) or isinstance(data[0], str) else data
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    embedding =[]
    for i in tqdm(np.arange(0, len(data), bs)):
        batch = data[i:i+bs]
        descriptions = [item["description"] for item in batch] 
        toks = tokenizer(
            descriptions,
            padding="max_length",
            max_length=40,
            truncation=True,
            return_tensors="pt"
        )
        
        tokse = {}
        for key,value in toks.items():
            tokse[key] = value.to(device)
        cls_rep = model(**tokse)[0][:,0,:] 
        embedding.append(cls_rep.cpu().detach())
    embedding = torch.cat(embedding)
    return embedding

In [111]:
def cos_sim(a, b):
    a_norm = torch.nn.functional.normalize(a, p=2, dim=1)
    b_norm = torch.nn.functional.normalize(b, p=2, dim=1)
    return torch.mm(a_norm, b_norm.transpose(0, 1))

# cosine similarity of first entity with all the entities



In [156]:
def max_log(embedded_database: torch.Tensor, embedded_input: torch.Tensor)-> int:
    similarity_tensor = cos_sim(embedded_database, embedded_input)
    log = torch.log(similarity_tensor).sum(dim=1)
    print(len(log))
    index= log.argmax().item()
    return(index)

In [138]:
bs = 32
all_embs = german_medical_embedding(icd_data, bs, tokenizer,model)


100%|██████████| 524/524 [00:17<00:00, 29.74it/s]


In [157]:
testing = cos_sim(all_embs[0].unsqueeze(0), all_embs)
print(testing)
print(np.argmax(testing))

tensor([[1.0000, 0.9283, 0.7229,  ..., 0.3239, 0.3501, 0.3125]])
tensor(0)


In [158]:
x='Typhus abdominalis und Paratyphus'
y=preprocess_embedding(x)
z = ['Sonstige', 'näher', 'bezeichnete', 'Karzinome', 'der', 'Leber']
print(x)

this_embed = german_medical_embedding (x, bs, tokenizer, model)
k= max_log(all_embs, this_embed)
print(k)
print(icd_data[k]['description'])
print(icd_data[k]['icdcode'])



Typhus abdominalis und Paratyphus


100%|██████████| 1/1 [00:00<00:00, 12.01it/s]

16755
326
Sonstige vorwiegend durch Geschlechtsverkehr übertragene Krankheiten, anderenorts nicht klassifiziert
A63


In [159]:
print(y)
this_embed = german_medical_embedding (y, bs, tokenizer, model)
k= max_log(all_embs, this_embed)
print(k)
print(icd_data[k]['description'])
print(icd_data[k]['icdcode'])

[{'description': 'Typhus'}, {'description': 'abdominalis'}, {'description': 'und'}, {'description': 'Paratyphus'}]


100%|██████████| 1/1 [00:00<00:00, 12.24it/s]

16755
326
Sonstige vorwiegend durch Geschlechtsverkehr übertragene Krankheiten, anderenorts nicht klassifiziert
A63


In [160]:
print(z)
this_embed = german_medical_embedding (z, bs, tokenizer, model)
k= max_log(all_embs, this_embed)
print(k)
print(icd_data[k]['description'])
print(icd_data[k]['icdcode'])

['Sonstige', 'näher', 'bezeichnete', 'Karzinome', 'der', 'Leber']


100%|██████████| 1/1 [00:00<00:00, 11.35it/s]

16755
3
Typhus abdominalis und Paratyphus
A01
